# 06_inference_contract_and_app_integration.ipynb — Validação e promoção condicional

## Objetivo
Validar um candidato treinado no Notebook 05 fora do contexto do treino, consolidar seu pacote de inferência e realizar uma promoção **condicional** para uso futuro no app/backend.

## Papel deste notebook
- Não treina modelo.
- Não escolhe campeão metodológico por si só.
- Apenas valida se um candidato já treinado:
  - carrega corretamente;
  - responde com contrato de inferência estável;
  - produz erros padronizados em inputs inválidos;
  - pode ser empacotado para handoff.

## Estratégia de promoção
- O notebook sempre cria um **pacote candidato validado**.
- A promoção para `active_model.json` só acontece se:
  1. o gate final passar;
  2. a flag `ALLOW_ACTIVE_PROMOTION = True` estiver ligada.

## Entradas esperadas
- artefatos do Notebook 05:
  - `models/05_train_real/<EXP_NAME>/best.pt`
  - `data/processed/05_runs/<EXP_NAME>/train_config.json`
  - `data/processed/05_runs/<EXP_NAME>/preprocess_config.json`
  - `data/processed/05_runs/<EXP_NAME>/inference_config.json`
  - `data/processed/05_runs/<EXP_NAME>/metrics_summary.json`
  - `data/processed/05_runs/<EXP_NAME>/test_metrics.json`
  - `data/processed/05_runs/<EXP_NAME>/final_summary.md`
- `data/processed/label_map.json`
- `data/processed/test.csv`

## Saídas
- `models/classification/candidates/<EXP_NAME>/...`
- `models/classification/validation_report.json`
- `models/classification/active_model.json` (somente promoção condicional)
- `reports/api_contract_example.json`
- `reports/integration_checklist.md`

In [1]:
# =========================
# 06_inference_contract_and_app_integration.ipynb — Célula 01
# Setup + imports + seed
# =========================

from __future__ import annotations

import os
import io
import json
import shutil
import random
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torchvision
import torchvision.models as tvm
import torchvision.transforms as T

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("OK — imports carregados.")
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

OK — imports carregados.
torch: 2.11.0.dev20260203+cu128
torchvision: 0.25.0.dev20260203+cu128
DEVICE: cuda
GPU: NVIDIA GeForce RTX 5080


In [2]:
# =========================
# 06_inference_contract_and_app_integration.ipynb — Célula 02
# PROJECT_ROOT + config principal
# =========================

def _looks_like_repo_root(p: Path) -> bool:
    return (
        (p / "data" / "processed").exists()
        and (p / "models").exists()
        and (p / "reports").exists()
    )

def _normalize_repo_candidate(p: Path) -> Optional[Path]:
    p = p.expanduser().resolve()
    if _looks_like_repo_root(p):
        return p
    if _looks_like_repo_root(p / "pimple"):
        return (p / "pimple").resolve()
    return None

def find_project_root_robust() -> Path:
    env_root = os.environ.get("PIMPLE_PROJECT_ROOT") or os.environ.get("PROJECT_ROOT")
    if env_root:
        normalized = _normalize_repo_candidate(Path(env_root))
        if normalized is not None:
            return normalized
        raise FileNotFoundError(
            f"[ERRO] Env PROJECT_ROOT/PIMPLE_PROJECT_ROOT aponta para {env_root}, "
            "mas não parece ser a raiz do repo pimple nem o diretório pai que contém a pasta pimple."
        )

    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        normalized = _normalize_repo_candidate(base)
        if normalized is not None:
            return normalized

    raise FileNotFoundError(
        "Não consegui localizar a raiz do repositório pimple.\n"
        "Dica: defina os.environ['PIMPLE_PROJECT_ROOT'] = r'CAMINHO_PARA_O_REPO_PIMPLE' e rode de novo."
    )

PROJECT_ROOT = find_project_root_robust()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

# candidato atual do Notebook 05
CANDIDATE_EXP_NAME = "cls_resnet50_img224_seed42_20260415_161053"

# promoção condicional: deixe False por padrão
ALLOW_ACTIVE_PROMOTION = False

# pasta final do pacote validado
CLASSIFICATION_DIR = MODELS_DIR / "classification"
CANDIDATES_DIR = CLASSIFICATION_DIR / "candidates"
CANDIDATE_PACKAGE_DIR = CANDIDATES_DIR / CANDIDATE_EXP_NAME

CLASSIFICATION_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATES_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATE_PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CANDIDATE_EXP_NAME:", CANDIDATE_EXP_NAME)
print("ALLOW_ACTIVE_PROMOTION:", ALLOW_ACTIVE_PROMOTION)
print("CANDIDATE_PACKAGE_DIR:", CANDIDATE_PACKAGE_DIR)

PROJECT_ROOT: C:\Users\win\Documents\GitHub\pimple
CANDIDATE_EXP_NAME: cls_resnet50_img224_seed42_20260415_161053
ALLOW_ACTIVE_PROMOTION: False
CANDIDATE_PACKAGE_DIR: C:\Users\win\Documents\GitHub\pimple\models\classification\candidates\cls_resnet50_img224_seed42_20260415_161053


In [3]:
# =========================
# 06_inference_contract_and_app_integration.ipynb — Célula 03
# Localizar artefatos de origem do Notebook 05
# =========================

SRC_MODEL_DIR = MODELS_DIR / "05_train_real" / CANDIDATE_EXP_NAME
SRC_RUN_DIR = PROCESSED_DIR / "05_runs" / CANDIDATE_EXP_NAME

BEST_PT = SRC_MODEL_DIR / "best.pt"
TRAIN_CONFIG_JSON = SRC_RUN_DIR / "train_config.json"
PREPROCESS_CONFIG_JSON = SRC_RUN_DIR / "preprocess_config.json"
INFERENCE_CONFIG_JSON = SRC_RUN_DIR / "inference_config.json"
METRICS_SUMMARY_JSON = SRC_RUN_DIR / "metrics_summary.json"
TEST_METRICS_JSON = SRC_RUN_DIR / "test_metrics.json"
FINAL_SUMMARY_MD = SRC_RUN_DIR / "final_summary.md"
ERROR_ANALYSIS_CSV = SRC_RUN_DIR / "error_analysis.csv"
LABEL_MAP_JSON = PROCESSED_DIR / "label_map.json"
TEST_CSV = PROCESSED_DIR / "test.csv"

required_sources = {
    "best.pt": BEST_PT,
    "train_config.json": TRAIN_CONFIG_JSON,
    "preprocess_config.json": PREPROCESS_CONFIG_JSON,
    "inference_config.json": INFERENCE_CONFIG_JSON,
    "metrics_summary.json": METRICS_SUMMARY_JSON,
    "test_metrics.json": TEST_METRICS_JSON,
    "final_summary.md": FINAL_SUMMARY_MD,
    "error_analysis.csv": ERROR_ANALYSIS_CSV,
    "label_map.json": LABEL_MAP_JSON,
    "test.csv": TEST_CSV,
}

missing = {k: str(v) for k, v in required_sources.items() if not v.exists()}
if missing:
    raise FileNotFoundError(
        "[ERRO] Artefatos obrigatórios ausentes para validar o candidato:\n"
        + json.dumps(missing, indent=2, ensure_ascii=False)
    )

print("OK — todos os artefatos obrigatórios foram encontrados.")
for k, v in required_sources.items():
    print(f"- {k}: {v}")

OK — todos os artefatos obrigatórios foram encontrados.
- best.pt: C:\Users\win\Documents\GitHub\pimple\models\05_train_real\cls_resnet50_img224_seed42_20260415_161053\best.pt
- train_config.json: C:\Users\win\Documents\GitHub\pimple\data\processed\05_runs\cls_resnet50_img224_seed42_20260415_161053\train_config.json
- preprocess_config.json: C:\Users\win\Documents\GitHub\pimple\data\processed\05_runs\cls_resnet50_img224_seed42_20260415_161053\preprocess_config.json
- inference_config.json: C:\Users\win\Documents\GitHub\pimple\data\processed\05_runs\cls_resnet50_img224_seed42_20260415_161053\inference_config.json
- metrics_summary.json: C:\Users\win\Documents\GitHub\pimple\data\processed\05_runs\cls_resnet50_img224_seed42_20260415_161053\metrics_summary.json
- test_metrics.json: C:\Users\win\Documents\GitHub\pimple\data\processed\05_runs\cls_resnet50_img224_seed42_20260415_161053\test_metrics.json
- final_summary.md: C:\Users\win\Documents\GitHub\pimple\data\processed\05_runs\cls_resnet

In [4]:
# =========================
# 06_inference_contract_and_app_integration.ipynb — Célula 04
# Carregar metadata do candidato + baseline de referência
# =========================

def json_load(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

train_config = json_load(TRAIN_CONFIG_JSON)
preprocess_config = json_load(PREPROCESS_CONFIG_JSON)
inference_config = json_load(INFERENCE_CONFIG_JSON)
metrics_summary = json_load(METRICS_SUMMARY_JSON)
test_metrics_payload = json_load(TEST_METRICS_JSON)
label_map = json_load(LABEL_MAP_JSON)

baseline_metrics_path = PROCESSED_DIR / "baseline_metrics.json"
baseline_metrics = json_load(baseline_metrics_path) if baseline_metrics_path.exists() else {}

test_df = pd.read_csv(TEST_CSV)

model_name = train_config["model_name"]
img_size = int(train_config["img_size"])
class_names = list(train_config["classes"])
num_classes = int(train_config["num_classes"])

val_metrics = metrics_summary.get("val", {})
test_metrics = metrics_summary.get("test", {})

baseline_majority_f1 = (
    baseline_metrics.get("baselines", {})
    .get("majority_class", {})
    .get("test", {})
    .get("f1_macro", None)
)

baseline_logreg_f1 = (
    baseline_metrics.get("baselines", {})
    .get("logreg_features", {})
    .get("test", {})
    .get("f1_macro", None)
)

print("model_name:", model_name)
print("img_size:", img_size)
print("num_classes:", num_classes)
print("class_names:", class_names)
print("val_metrics:", val_metrics)
print("test_metrics:", test_metrics)
print("baseline_majority_test_f1_macro:", baseline_majority_f1)
print("baseline_logreg_test_f1_macro:", baseline_logreg_f1)

model_name: resnet50
img_size: 224
num_classes: 7
class_names: ['mel', 'nv', 'bcc', 'akiec', 'bkl', 'df', 'vasc']
val_metrics: {'accuracy': 0.8715046604527297, 'f1_macro': 0.7666367637736983}
test_metrics: {'accuracy': 0.8628495339547271, 'f1_macro': 0.7681522012145432}
baseline_majority_test_f1_macro: 0.11460469355206197
baseline_logreg_test_f1_macro: 0.33904869510914043


In [5]:
# =========================
# 06_inference_contract_and_app_integration.ipynb — Célula 05
# Consolidar pacote candidato validável
# =========================

PACKAGE_FILES_TO_COPY = {
    "best.pt": BEST_PT,
    "train_config.json": TRAIN_CONFIG_JSON,
    "preprocess_config.json": PREPROCESS_CONFIG_JSON,
    "inference_config.json": INFERENCE_CONFIG_JSON,
    "metrics_summary.json": METRICS_SUMMARY_JSON,
    "test_metrics.json": TEST_METRICS_JSON,
    "final_summary.md": FINAL_SUMMARY_MD,
    "error_analysis.csv": ERROR_ANALYSIS_CSV,
    "label_map.json": LABEL_MAP_JSON,
}

for filename, src in PACKAGE_FILES_TO_COPY.items():
    dst = CANDIDATE_PACKAGE_DIR / filename
    shutil.copy2(src, dst)

print("OK — pacote candidato copiado para:")
for filename in PACKAGE_FILES_TO_COPY:
    print("-", CANDIDATE_PACKAGE_DIR / filename)

OK — pacote candidato copiado para:
- C:\Users\win\Documents\GitHub\pimple\models\classification\candidates\cls_resnet50_img224_seed42_20260415_161053\best.pt
- C:\Users\win\Documents\GitHub\pimple\models\classification\candidates\cls_resnet50_img224_seed42_20260415_161053\train_config.json
- C:\Users\win\Documents\GitHub\pimple\models\classification\candidates\cls_resnet50_img224_seed42_20260415_161053\preprocess_config.json
- C:\Users\win\Documents\GitHub\pimple\models\classification\candidates\cls_resnet50_img224_seed42_20260415_161053\inference_config.json
- C:\Users\win\Documents\GitHub\pimple\models\classification\candidates\cls_resnet50_img224_seed42_20260415_161053\metrics_summary.json
- C:\Users\win\Documents\GitHub\pimple\models\classification\candidates\cls_resnet50_img224_seed42_20260415_161053\test_metrics.json
- C:\Users\win\Documents\GitHub\pimple\models\classification\candidates\cls_resnet50_img224_seed42_20260415_161053\final_summary.md
- C:\Users\win\Documents\GitHub\

In [6]:
# =========================
# 06_inference_contract_and_app_integration.ipynb — Célula 06
# Loader de modelo + preprocess de inferência
# =========================

def build_torchvision_model(name: str, num_classes: int) -> nn.Module:
    name = name.lower()

    if name == "resnet50":
        m = tvm.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m

    if name == "efficientnet_b0":
        m = tvm.efficientnet_b0(weights=None)
        in_f = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_f, num_classes)
        return m

    if name == "mobilenet_v3_large":
        m = tvm.mobilenet_v3_large(weights=None)
        in_f = m.classifier[3].in_features
        m.classifier[3] = nn.Linear(in_f, num_classes)
        return m

    raise ValueError(
        f"MODEL_NAME inválido: {name}. Use resnet50 / efficientnet_b0 / mobilenet_v3_large."
    )

ckpt = torch.load(CANDIDATE_PACKAGE_DIR / "best.pt", map_location=DEVICE)
model = build_torchvision_model(model_name, num_classes)
model.load_state_dict(ckpt["state_dict"])
model.to(DEVICE)
model.eval()

mean = preprocess_config.get("normalize_mean", [0.485, 0.456, 0.406])
std = preprocess_config.get("normalize_std", [0.229, 0.224, 0.225])

infer_tfms = T.Compose([
    T.Resize((img_size, img_size)),
    T.ToTensor(),
    T.Normalize(mean=mean, std=std),
])

print("OK — checkpoint carregado com sucesso.")
print("ckpt epoch:", ckpt.get("epoch"))
print("ckpt val_f1_macro:", ckpt.get("val_f1_macro"))

OK — checkpoint carregado com sucesso.
ckpt epoch: 18
ckpt val_f1_macro: 0.7666367637736983


In [7]:
# =========================
# 06_inference_contract_and_app_integration.ipynb — Célula 07
# Contrato de inferência: helpers
# =========================

def success_contract(
    probs: np.ndarray,
    class_names: List[str],
    model_version: str,
    img_size: int,
    top_k_default: int = 3,
) -> Dict[str, Any]:
    probs = np.asarray(probs, dtype=np.float32)
    top_k = min(int(top_k_default), len(class_names))

    order = np.argsort(-probs)[:top_k]
    top_pred_idx = int(order[0])

    return {
        "task": "classification",
        "model_version": model_version,
        "top_prediction": {
            "label": class_names[top_pred_idx],
            "score": float(probs[top_pred_idx]),
        },
        "top_k": [
            {
                "label": class_names[int(i)],
                "score": float(probs[int(i)]),
            }
            for i in order
        ],
        "preprocess": {
            "size": [int(img_size), int(img_size)],
            "normalize_mean": list(mean),
            "normalize_std": list(std),
        },
    }

def error_contract(code: str, message: str) -> Dict[str, Any]:
    return {
        "error": {
            "code": str(code),
            "message": str(message),
        }
    }

def infer_pil_image(img: Image.Image, top_k_default: int = 3) -> Dict[str, Any]:
    if img is None:
        return error_contract("INVALID_INPUT", "Imagem ausente.")

    if not isinstance(img, Image.Image):
        return error_contract("INVALID_INPUT", "Objeto de imagem inválido.")

    try:
        img = img.convert("RGB")
        x = infer_tfms(img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logits = model(x)
            probs = torch.softmax(logits, dim=1).detach().cpu().numpy()[0]

        return success_contract(
            probs=probs,
            class_names=class_names,
            model_version=CANDIDATE_EXP_NAME,
            img_size=img_size,
            top_k_default=inference_config.get("top_k_default", 3),
        )
    except Exception as e:
        return error_contract("INFERENCE_FAILED", f"Falha na inferência: {e}")

def infer_from_path(path: Path) -> Dict[str, Any]:
    if path is None or str(path).strip() == "":
        return error_contract("INVALID_INPUT", "Arquivo inválido ou caminho vazio.")
    path = Path(path)
    if not path.exists() or not path.is_file():
        return error_contract("INVALID_INPUT", "Arquivo inválido ou inexistente.")
    try:
        with Image.open(path) as im:
            return infer_pil_image(im)
    except Exception:
        return error_contract("INVALID_INPUT", "Arquivo inválido ou formato não suportado.")

In [8]:
# =========================
# 06_inference_contract_and_app_integration.ipynb — Célula 08
# Smoke test com imagem válida
# =========================

assert "image_path" in test_df.columns, "test.csv precisa ter coluna image_path"

sample_row = test_df.iloc[0].copy()
sample_image_path = (PROJECT_ROOT / str(sample_row["image_path"])).resolve()

if not sample_image_path.exists():
    raise FileNotFoundError(f"Imagem de smoke test não encontrada: {sample_image_path}")

valid_response = infer_from_path(sample_image_path)

print("sample_image_path:", sample_image_path)
print(json.dumps(valid_response, indent=2, ensure_ascii=False))

# gate mínimo do contrato de sucesso
success_contract_ok = (
    isinstance(valid_response, dict)
    and "task" in valid_response
    and "model_version" in valid_response
    and "top_prediction" in valid_response
    and "top_k" in valid_response
    and "preprocess" in valid_response
    and "error" not in valid_response
)

assert success_contract_ok, "Contrato de sucesso inválido no smoke test."
print("OK — smoke test válido.")

sample_image_path: C:\Users\win\Documents\GitHub\pimple\data\raw\lesions\images\ISIC_0024869.jpg
{
  "task": "classification",
  "model_version": "cls_resnet50_img224_seed42_20260415_161053",
  "top_prediction": {
    "label": "mel",
    "score": 0.4385339021682739
  },
  "top_k": [
    {
      "label": "mel",
      "score": 0.4385339021682739
    },
    {
      "label": "nv",
      "score": 0.22441184520721436
    },
    {
      "label": "df",
      "score": 0.10974466800689697
    }
  ],
  "preprocess": {
    "size": [
      224,
      224
    ],
    "normalize_mean": [
      0.485,
      0.456,
      0.406
    ],
    "normalize_std": [
      0.229,
      0.224,
      0.225
    ]
  }
}
OK — smoke test válido.


In [9]:
# =========================
# 06_inference_contract_and_app_integration.ipynb — Célula 09
# Testes de erro / input inválido
# =========================

invalid_none_response = infer_from_path(Path(""))
invalid_missing_response = infer_from_path(PROJECT_ROOT / "nao_existe.png")

fake_txt_path = CANDIDATE_PACKAGE_DIR / "_invalid_input.txt"
fake_txt_path.write_text("isto não é uma imagem", encoding="utf-8")
invalid_text_response = infer_from_path(fake_txt_path)

print("invalid_none_response:")
print(json.dumps(invalid_none_response, indent=2, ensure_ascii=False))

print("\ninvalid_missing_response:")
print(json.dumps(invalid_missing_response, indent=2, ensure_ascii=False))

print("\ninvalid_text_response:")
print(json.dumps(invalid_text_response, indent=2, ensure_ascii=False))

def is_error_contract(resp: Dict[str, Any]) -> bool:
    return isinstance(resp, dict) and "error" in resp and "code" in resp["error"] and "message" in resp["error"]

error_contracts_ok = (
    is_error_contract(invalid_none_response)
    and is_error_contract(invalid_missing_response)
    and is_error_contract(invalid_text_response)
)

assert error_contracts_ok, "Contrato de erro inválido em input inválido."
print("OK — contratos de erro válidos.")

invalid_none_response:
{
  "error": {
    "code": "INVALID_INPUT",
    "message": "Arquivo inválido ou inexistente."
  }
}

invalid_missing_response:
{
  "error": {
    "code": "INVALID_INPUT",
    "message": "Arquivo inválido ou inexistente."
  }
}

invalid_text_response:
{
  "error": {
    "code": "INVALID_INPUT",
    "message": "Arquivo inválido ou formato não suportado."
  }
}
OK — contratos de erro válidos.


In [10]:
# =========================
# 06_inference_contract_and_app_integration.ipynb — Célula 10
# model_card.md + example_input_output.json + API contract example
# =========================

model_card_md = f"""# Model Card — {CANDIDATE_EXP_NAME}

## Status
- status: candidate_v1_validated
- official_promotion: conditional
- active_model: {str(ALLOW_ACTIVE_PROMOTION)}

## Run
- exp_name: `{CANDIDATE_EXP_NAME}`
- model_name: `{model_name}`
- num_classes: `{num_classes}`
- classes: `{", ".join(class_names)}`

## Metrics
- val_accuracy: `{val_metrics.get("accuracy")}`
- val_f1_macro: `{val_metrics.get("f1_macro")}`
- test_accuracy: `{test_metrics.get("accuracy")}`
- test_f1_macro: `{test_metrics.get("f1_macro")}`

## Intended use
Modelo educacional/portfolio para classificação de imagens de lesões cutâneas.

## Important limitation
Este modelo **não é clínico**, **não é dispositivo médico** e **não deve ser usado para decisão médica real**.

## Notes
- O candidato foi validado fora do Notebook 05.
- A promoção para modelo ativo depende do gate final deste Notebook 06.
- Se houver nova comparação metodológica entre candidatos, este pacote pode deixar de ser o principal.
"""

example_input_output = {
    "input_example": {
        "image_path": str(sample_image_path.relative_to(PROJECT_ROOT)),
    },
    "output_example": valid_response,
    "error_example": invalid_text_response,
}

api_contract_example = {
    "success": valid_response,
    "error": invalid_text_response,
}

(CANDIDATE_PACKAGE_DIR / "model_card.md").write_text(model_card_md, encoding="utf-8")
(CANDIDATE_PACKAGE_DIR / "example_input_output.json").write_text(
    json.dumps(example_input_output, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

(REPORTS_DIR / "api_contract_example.json").write_text(
    json.dumps(api_contract_example, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("OK — model_card.md salvo.")
print("OK — example_input_output.json salvo.")
print("OK — reports/api_contract_example.json salvo.")

OK — model_card.md salvo.
OK — example_input_output.json salvo.
OK — reports/api_contract_example.json salvo.


In [11]:
# =========================
# 06_inference_contract_and_app_integration.ipynb — Célula 11
# Gate de validação e promoção condicional
# =========================

val_f1 = float(val_metrics.get("f1_macro", 0.0))
test_f1 = float(test_metrics.get("f1_macro", 0.0))

beats_logreg = True if baseline_logreg_f1 is None else (test_f1 > float(baseline_logreg_f1))
beats_majority = True if baseline_majority_f1 is None else (test_f1 > float(baseline_majority_f1))

gate_checks = {
    "best_pt_exists": BEST_PT.exists(),
    "train_config_exists": TRAIN_CONFIG_JSON.exists(),
    "preprocess_config_exists": PREPROCESS_CONFIG_JSON.exists(),
    "inference_config_exists": INFERENCE_CONFIG_JSON.exists(),
    "metrics_summary_exists": METRICS_SUMMARY_JSON.exists(),
    "test_metrics_exists": TEST_METRICS_JSON.exists(),
    "final_summary_exists": FINAL_SUMMARY_MD.exists(),
    "error_analysis_exists": ERROR_ANALYSIS_CSV.exists(),
    "label_map_exists": LABEL_MAP_JSON.exists(),
    "checkpoint_loaded": True,
    "success_contract_ok": success_contract_ok,
    "error_contracts_ok": error_contracts_ok,
    "val_f1_macro_min_0_60": val_f1 >= 0.60,
    "test_f1_macro_min_0_60": test_f1 >= 0.60,
    "beats_majority_baseline": beats_majority,
    "beats_logreg_baseline": beats_logreg,
    "candidate_package_dir_exists": CANDIDATE_PACKAGE_DIR.exists(),
    "model_card_exists": (CANDIDATE_PACKAGE_DIR / "model_card.md").exists(),
    "example_input_output_exists": (CANDIDATE_PACKAGE_DIR / "example_input_output.json").exists(),
    "api_contract_example_exists": (REPORTS_DIR / "api_contract_example.json").exists(),
}

gate_pass = all(gate_checks.values())

validation_report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "candidate_exp_name": CANDIDATE_EXP_NAME,
    "allow_active_promotion": bool(ALLOW_ACTIVE_PROMOTION),
    "gate_pass": bool(gate_pass),
    "checks": gate_checks,
    "metrics": {
        "val": val_metrics,
        "test": test_metrics,
        "baseline_majority_test_f1_macro": baseline_majority_f1,
        "baseline_logreg_test_f1_macro": baseline_logreg_f1,
    },
    "source_paths": {k: str(v) for k, v in required_sources.items()},
    "package_dir": str(CANDIDATE_PACKAGE_DIR),
}

validation_report_path = CLASSIFICATION_DIR / "validation_report.json"
validation_report_path.write_text(
    json.dumps(validation_report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

active_model_path = CLASSIFICATION_DIR / "active_model.json"
promoted = False

if gate_pass and ALLOW_ACTIVE_PROMOTION:
    active_payload = {
        "activated_at_utc": datetime.now(timezone.utc).isoformat(),
        "status": "active",
        "model_family": "classification",
        "exp_name": CANDIDATE_EXP_NAME,
        "package_dir": str(CANDIDATE_PACKAGE_DIR),
        "model_name": model_name,
        "metrics": {
            "val": val_metrics,
            "test": test_metrics,
        },
        "note": "Promoção condicional realizada pelo Notebook 06 após gate PASS.",
    }
    active_model_path.write_text(
        json.dumps(active_payload, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    promoted = True

print("validation_report:", validation_report_path)
print("gate_pass:", gate_pass)
print("promoted_to_active:", promoted)
if active_model_path.exists():
    print("active_model.json:", active_model_path)

validation_report: C:\Users\win\Documents\GitHub\pimple\models\classification\validation_report.json
gate_pass: True
promoted_to_active: False


In [12]:
# =========================
# 06_inference_contract_and_app_integration.ipynb — Célula 12
# Checklist final de integração + resumo executivo
# =========================

integration_checklist_md = f"""# Integration Checklist — {CANDIDATE_EXP_NAME}

## Candidate
- exp_name: `{CANDIDATE_EXP_NAME}`
- model_name: `{model_name}`
- package_dir: `{CANDIDATE_PACKAGE_DIR}`

## Validation status
- gate_pass: `{gate_pass}`
- allow_active_promotion: `{ALLOW_ACTIVE_PROMOTION}`
- promoted_to_active: `{promoted}`

## Smoke tests
- success_contract_ok: `{success_contract_ok}`
- error_contracts_ok: `{error_contracts_ok}`

## Metrics
- val_accuracy: `{val_metrics.get("accuracy")}`
- val_f1_macro: `{val_metrics.get("f1_macro")}`
- test_accuracy: `{test_metrics.get("accuracy")}`
- test_f1_macro: `{test_metrics.get("f1_macro")}`

## Package files
- best.pt
- train_config.json
- preprocess_config.json
- inference_config.json
- metrics_summary.json
- test_metrics.json
- final_summary.md
- error_analysis.csv
- label_map.json
- model_card.md
- example_input_output.json

## Contract
- success example: `reports/api_contract_example.json`
- error example: `reports/api_contract_example.json`

## Recommendation
{"- Candidate validado e promovido para active_model.json." if promoted else "- Candidate validado como pacote candidato. Promoção ativa ainda não realizada."}

## Important note
Este pacote continua sendo **educacional** e **não clínico**.
"""

integration_checklist_path = REPORTS_DIR / "integration_checklist.md"
integration_checklist_path.write_text(integration_checklist_md, encoding="utf-8")

print("=== Resumo final ===")
print("candidate:", CANDIDATE_EXP_NAME)
print("package_dir:", CANDIDATE_PACKAGE_DIR)
print("validation_report:", validation_report_path)
print("integration_checklist:", integration_checklist_path)
print("gate_pass:", gate_pass)
print("promoted_to_active:", promoted)

print("\n=== Gate final ===")
for k, v in gate_checks.items():
    print(f"- {k}: {v}")
print("STATUS:", "PASS" if gate_pass else "FAIL")

=== Resumo final ===
candidate: cls_resnet50_img224_seed42_20260415_161053
package_dir: C:\Users\win\Documents\GitHub\pimple\models\classification\candidates\cls_resnet50_img224_seed42_20260415_161053
validation_report: C:\Users\win\Documents\GitHub\pimple\models\classification\validation_report.json
integration_checklist: C:\Users\win\Documents\GitHub\pimple\reports\integration_checklist.md
gate_pass: True
promoted_to_active: False

=== Gate final ===
- best_pt_exists: True
- train_config_exists: True
- preprocess_config_exists: True
- inference_config_exists: True
- metrics_summary_exists: True
- test_metrics_exists: True
- final_summary_exists: True
- error_analysis_exists: True
- label_map_exists: True
- checkpoint_loaded: True
- success_contract_ok: True
- error_contracts_ok: True
- val_f1_macro_min_0_60: True
- test_f1_macro_min_0_60: True
- beats_majority_baseline: True
- beats_logreg_baseline: True
- candidate_package_dir_exists: True
- model_card_exists: True
- example_input_o